In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("Powerplant_dataset.csv")

In [6]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [ ]:
# AT - temperature
# V - vaccum
# AP - pressure
# RH - humadity
# PE - produced energy


In [7]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [9]:
X = df.drop("PE", axis=1)
y = df["PE"]

In [10]:
# split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
df.shape

(9568, 5)

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [18]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [19]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Deep Learning

In [21]:
# Define ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
    
        self.model = nn.Sequential(
            # 1st Hidden Layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # output Layer
            nn.Linear(6, 1),
        )
    
    def forward(self, x):
        return self.model(x)

In [23]:
import torch.optim as optim
model = ANN()

# loss, optimizer

crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# train the ANN
epochs = 100

for epoch in range(epochs):
    model.train()
    runnig_loss = 0.0 # total training loss for 1 epoch

    for xb, yb in train_loader:
        # xb = feature of 1 batch
        # yb = labels of 1 batch
        outputs = model(xb) # forward prop.... predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # back prop.... compute gradients
        optimizer.step() # params update

        runnig_loss += loss